# 03 · End-to-end Pipeline (100 seq)

**Purpose.** Validate the *full* pipeline (routing → alias resolve → direct/tblastn lift → output), not just the lifting engine. Reproduces `e2e_summary.tsv` / `e2e_accuracy.png`.

## Inputs

FMD + PRRS ref/query pairs.

In [ ]:
from pathlib import Path
import sys, time

import pandas as pd
import matplotlib.pyplot as plt

# --- anchor ROOT to the repo (folder that contains app/src) ---
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA   = ROOT / "app" / "data"
CONFIG = ROOT / "app" / "config"

# References (verify these are the intended ref records for the paper):
FMD_REF    = DATA / "FMD"  / "FMD_ref_test.gb"        # alt: FMD_FJ175661_Anno.gb
FMD_QUERY  = DATA / "FMD"  / "FMD_100seq_anno.gb"
PRRS_REF   = DATA / "PRRS" / "PRRS_ref_test.gb"       # alt: PRRS_MT746146_Anno.gb
PRRS_QUERY = DATA / "PRRS" / "PRRS_100seq_anno.gb"
PED_REFS   = {"ref_1": DATA / "PED" / "PED_ref_1.gb",
              "ref_2": DATA / "PED" / "PED_ref_2.gb"}
PED_QUERY  = DATA / "PED" / "PED_100seqs.gb"

# Run toggle: keep False for a fast smoke test, True for the full 100-record run.
RUN_FULL = False
SAMPLE_N = 10


In [ ]:
# outputs land inside this unit folder so figures/tables sit next to the notebook
UNIT_DIR = ROOT / "app" / "validation" / "10_pipeline_end_to_end"
OUT = UNIT_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## Run — production pipeline against truth (real module)

In [ ]:
from app.validation._shared.validation_utils import run_production_pipeline_against_truth, summarize_comparison

pairs = {"fmd": (FMD_REF, FMD_QUERY), "prrs": (PRRS_REF, PRRS_QUERY)}
e2e = {}
for label, (ref, query) in pairs.items():
    per_pred, summary = run_production_pipeline_against_truth(label, ref, query, OUT / label)
    per_pred["virus"] = label
    e2e[label] = per_pred
e2e_all = pd.concat(e2e.values(), ignore_index=True)
e2e_all.to_csv(OUT / "e2e_per_prediction.tsv", sep="\t", index=False)
e2e_all.head()

## Metrics

Status mix (exact / coord / failed) and overall accuracy per virus, plus which route (direct vs tblastn) each prediction took.

In [ ]:
status_summary = (e2e_all.groupby(["virus"])
                  .agg(total=("pred_name", "size"),
                       exact=("exact_match", "sum"),
                       coord_correct=("coord_correct", "sum"))
                  .reset_index())
status_summary["correct_pct"] = ((status_summary["exact"] + status_summary["coord_correct"])
                                 / status_summary["total"] * 100).round(2)
status_summary.to_csv(OUT / "e2e_summary.tsv", sep="\t", index=False)
status_summary

## Figure

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(status_summary["virus"], status_summary["correct_pct"])
ax.set_ylabel("correct %"); ax.set_ylim(0, 100); ax.set_title("End-to-end accuracy")
fig.tight_layout(); fig.savefig(OUT / "e2e_accuracy.png", dpi=200)

## Interpretation

> ⚠️ **TODO**: pipeline hoạt động end-to-end; routing chọn đúng direct/tblastn; sai số còn lại chuyển sang phân tích ở unit 04.